# Neural Network Dreams Bad Apple — Colab anchor ablation

This notebook runs one V4.2 anchor-budget experiment at a time on a hosted Colab GPU while you edit the notebook in VS Code. It does **not** start training when opened.

The Colab kernel cannot read `D:\\Code_archive\\Bad_apple` directly. To keep the repository private, this notebook reads a small source bundle plus the video and frozen autoencoder from Google Drive. From the local repository root, create the source bundle in PowerShell:

```powershell
Compress-Archive -Path prototype.py,requirements.txt,README.md,neural_bad_apple,tests -DestinationPath bad_apple_code.zip -Force
```

Then put these three files in `MyDrive/neural_bad_apple/assets/`:

- `bad_apple_code.zip`
- `bad_apple.mp4`
- `autoencoder_model_best.pt` (the existing `prototype_runs/basic_full/model_best.pt`)

Checkpoints, metrics, and the per-run `report.md` are written directly to Drive so a Colab disconnect does not erase completed epochs. Training itself uses `/content` for the frame dataset because reading thousands of PNGs directly from Drive is slow.

In [ ]:
# Configuration — edit these values before running the setup cells.
from pathlib import Path

PROJECT_DIR = Path("/content/bad_apple")
DRIVE_ROOT = Path("/content/drive/MyDrive/neural_bad_apple")
CODE_ARCHIVE = DRIVE_ROOT / "assets/bad_apple_code.zip"
VIDEO_SOURCE = DRIVE_ROOT / "assets/bad_apple.mp4"
AUTOENCODER_SOURCE = DRIVE_ROOT / "assets/autoencoder_model_best.pt"

ANCHORS = 0          # Recommended first pass: 0, then 32; 220 already exists locally.
EPOCHS = 12
TRAIN_BATCH_SIZE = 2 # Keep 2 for direct comparability with the local 220-anchor run.
EVAL_BATCH_SIZE = 16
SEED = 7

assert ANCHORS in {0, 16, 32, 55, 110}, "Choose a staged, not-yet-trained budget."


In [ ]:
# Runtime check. In VS Code: Select Kernel -> Colab -> choose the A100 runtime.
import platform
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select a Colab GPU runtime first.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {gpu_name} ({gpu_memory_gib:.1f} GiB)")
if "A100" not in gpu_name.upper():
    print("Warning: this is not an A100; the experiment will still work but take longer.")


In [ ]:
# Mount persistent storage. The VS Code extension may open a browser authorization prompt.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Persistent experiment root: {DRIVE_ROOT}")


In [ ]:
# Unpack the private source bundle and install the project's small dependencies.
import os
import shutil
import subprocess

def run(command, *, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)

if not CODE_ARCHIVE.is_file():
    raise FileNotFoundError(f"Create and upload the private source bundle to {CODE_ARCHIVE}")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(CODE_ARCHIVE, PROJECT_DIR)
if not (PROJECT_DIR / "prototype.py").is_file():
    raise RuntimeError("The archive has an extra parent folder. Recreate it with the PowerShell command above.")

run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f"Private source ready from {CODE_ARCHIVE} ({CODE_ARCHIVE.stat().st_size / 2**20:.1f} MiB)")


In [ ]:
# Stage the frozen autoencoder on Colab's local disk.
import shutil

if not VIDEO_SOURCE.is_file():
    raise FileNotFoundError(f"Upload the source video to {VIDEO_SOURCE}")
if not AUTOENCODER_SOURCE.is_file():
    raise FileNotFoundError(f"Upload the frozen autoencoder checkpoint to {AUTOENCODER_SOURCE}")

autoencoder_local = PROJECT_DIR / "prototype_runs/basic_full/model_best.pt"
autoencoder_local.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUTOENCODER_SOURCE, autoencoder_local)
print(f"Autoencoder ready: {autoencoder_local}")


In [ ]:
# Extract the full 219.1-second source locally. This is skipped when all frames already exist.
frame_dir = PROJECT_DIR / "prototype_data/full_source_frames"
frame_count = len(list(frame_dir.glob("frame_*.png"))) if frame_dir.exists() else 0

if frame_count != 6573:
    run([
        "python", "prototype.py", "extract",
        "--input", VIDEO_SOURCE,
        "--output-dir", frame_dir,
        "--manifest", PROJECT_DIR / "prototype_data/full_manifest.json",
        "--start", "0", "--end", "219.1", "--fps", "30", "--force",
    ], cwd=PROJECT_DIR)

frame_count = len(list(frame_dir.glob("frame_*.png")))
if frame_count != 6573:
    raise RuntimeError(f"Expected 6573 frames, found {frame_count}.")
print(f"Dataset ready: {frame_count} frames in {frame_dir}")


In [ ]:
# Fast preflight: catches a stale branch that lacks the zero-anchor implementation.
run(["python", "-m", "unittest", "discover", "-s", "tests", "-p", "test_pipeline.py", "-v"], cwd=PROJECT_DIR)


In [ ]:
# Build the paths and command for the selected budget. This cell only prints; it does not train.
variant = f"anchors_{ANCHORS:03d}"
run_dir = DRIVE_ROOT / "prototype_runs/anchor_budget_ablation" / variant
raw_output_dir = DRIVE_ROOT / "prototype_outputs/anchor_budget_ablation" / f"{variant}_raw"
polarity_run_dir = DRIVE_ROOT / "prototype_runs/anchor_budget_ablation" / f"{variant}_polarity"
final_output_dir = DRIVE_ROOT / "prototype_outputs/anchor_budget_ablation" / f"{variant}_final"

train_command = [
    "python", "prototype.py", "train-hybrid-v42",
    "--anchors", str(ANCHORS),
    "--run-dir", run_dir,
    "--epochs", str(EPOCHS),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--seed", str(SEED),
    "--device", "cuda",
]
print("Selected experiment:", variant)
print("Training output:", run_dir)
print("Command:", " ".join(map(str, train_command)))
print("No training has started.")


## Start one training run

Set `START_TRAINING = True` only when the printed budget and paths are correct. The trainer saves `model_last.pt`, `model_best.pt`, `history.json`, and an updated `report.md` after every completed epoch. It does not currently resume an interrupted partial run, so keep the runtime connected through the selected model.

In [ ]:
START_TRAINING = False

if not START_TRAINING:
    raise RuntimeError("Training is armed but disabled. Set START_TRAINING = True in this cell.")
if (run_dir / "model_best.pt").exists():
    raise FileExistsError(f"A checkpoint already exists at {run_dir}; refusing to overwrite it.")

run(train_command, cwd=PROJECT_DIR)


## Evaluate and repair polarity

Run this after training completes. It performs the raw rollout, fits the separate polarity spline, and performs the final metrics-only rollout.

In [ ]:
checkpoint = run_dir / "model_best.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(f"Training checkpoint not found: {checkpoint}")

run([
    "python", "prototype.py", "rollout-ar",
    "--checkpoint", checkpoint,
    "--data-dir", frame_dir,
    "--output-dir", raw_output_dir,
    "--batch-size", str(EVAL_BATCH_SIZE),
    "--device", "cuda", "--fps", "30", "--no-video",
], cwd=PROJECT_DIR)

run([
    "python", "prototype.py", "fix-polarity",
    "--checkpoint", checkpoint,
    "--target-csv", raw_output_dir / "error_curve.csv",
    "--run-dir", polarity_run_dir,
    "--device", "cuda",
], cwd=PROJECT_DIR)

run([
    "python", "prototype.py", "rollout-ar",
    "--checkpoint", polarity_run_dir / "model_best.pt",
    "--data-dir", frame_dir,
    "--output-dir", final_output_dir,
    "--batch-size", str(EVAL_BATCH_SIZE),
    "--device", "cuda", "--fps", "30", "--no-video",
], cwd=PROJECT_DIR)


In [ ]:
# Compact result readout for the blog/report table.
import json

summary_path = final_output_dir / "drift_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
result = {
    "anchors": ANCHORS,
    "rollout_error": summary["post_cutoff_mean_rollout_binary_error"],
    "accumulation_gap": summary["post_cutoff_mean_accumulation_gap"],
    "iou": summary["post_cutoff_mean_rollout_iou"],
    "peak_error": summary["peak_rollout_binary_error"],
    "peak_seconds": summary["peak_error_seconds"],
}
print(json.dumps(result, indent=2))
print(f"Full summary: {summary_path}")
print(f"Training report: {run_dir / 'report.md'}")


## Next budget

Recommended order is `0`, `32`, then compare both with the existing local `220` reference. If that establishes a useful trend, run `16`, `55`, and `110`. To start another budget, change `ANCHORS` in the configuration cell and rerun from the path/command cell downward.